In [44]:
import os

BOOKS_DIR = "books"
os.makedirs(BOOKS_DIR, exist_ok=True)


In [45]:
import pandas as pd
import requests

CSV_URL = "https://raw.githubusercontent.com/alexeygrigorev/ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv"

def download_books():
    df = pd.read_csv(CSV_URL)

    for _, row in df.iterrows():
        url = row["pdf_url"]
        filename = url.split("/")[-1]
        filepath = os.path.join(BOOKS_DIR, filename)

        if os.path.exists(filepath):
            print(f"Already exists: {filename}")
            continue

        print(f"Downloading: {filename}")
        response = requests.get(url)
        response.raise_for_status()

        with open(filepath, "wb") as f:
            f.write(response.content)

    print("All books downloaded")


In [46]:
download_books()

Already exists: thinkpython2.pdf
Already exists: thinkdsp.pdf
Already exists: thinkcomplexity2.pdf
Already exists: thinkjava2.pdf
Already exists: PhysicalModelingInMatlab4.pdf
Already exists: thinkos.pdf
Already exists: Think-C.pdf
All books downloaded


###  Markdown Conversion 

In [47]:
TEXT_DIR = "books_text"
os.makedirs(TEXT_DIR, exist_ok=True)


In [48]:
from markitdown import MarkItDown

md = MarkItDown()

def convert_pdfs_to_markdown():
    for filename in os.listdir(BOOKS_DIR):
        if not filename.endswith(".pdf"):
            continue

        pdf_path = os.path.join(BOOKS_DIR, filename)
        md_path = os.path.join(TEXT_DIR, filename.replace(".pdf", ".md"))

        print(f"Converting {filename} to markdown")
        result = md.convert(pdf_path)

        with open(md_path, "w", encoding="utf-8") as f:
            f.write(result.text_content)

    print("All PDFs converted to markdown")


In [49]:
convert_pdfs_to_markdown()


Converting PhysicalModelingInMatlab4.pdf to markdown
Converting Think-C.pdf to markdown
Converting thinkcomplexity2.pdf to markdown
Converting thinkdsp.pdf to markdown
Converting thinkjava2.pdf to markdown
Converting thinkos.pdf to markdown
Converting thinkpython2.pdf to markdown
All PDFs converted to markdown


In [59]:
!wc -l books_text/thinkpython.md


16268 books_text/thinkpython.md


### Reading files 

In [60]:
documents = []

for filename in os.listdir(TEXT_DIR):
    if not filename.endswith(".md"):
        continue

    path = os.path.join(TEXT_DIR, filename)

    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()


    cleaned_lines = [line.strip() for line in lines if line.strip()]

    documents.append({
        "source": filename,
        "content": cleaned_lines
    })


## Chunking the Documents

In [61]:
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=100,
    step=50
)


In [62]:
len([c for c in chunks if c["source"] == "thinkpython.md"])


214

## Indexing with minsearch

In [63]:
def prepare_documents(chunks):
    docs = []
    for chunk in chunks:
        docs.append({
            "source": chunk["source"],
            "content": "\n".join(chunk["content"])
        })
    return docs

indexed_documents = prepare_documents(chunks)


In [67]:
from minsearch import Index

index = Index(text_fields=["content"])
index.fit(indexed_documents)



In [68]:
len(indexed_documents)


919

In [69]:
results = index.search("python function definition", num_results=5)
results[0]["source"]


'thinkpython.md'

## RAG 

In [70]:
import json
from openai import OpenAI

openai_client = OpenAI()

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(question=question, context=context)

def rag_unstructured(query):
    search_results = index.search(query, num_results=5)
    prompt = build_prompt(query, search_results)

    response = openai_client.responses.create(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": prompt}
        ]
    )
    return response


In [71]:
unstructured_response = rag_unstructured("python function definition")
unstructured_response.usage.input_tokens


6834

## structured output 

In [72]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="Main answer in markdown")
    found_answer: bool
    confidence: float
    confidence_explanation: str
    answer_type: Literal[
        "how-to", "explanation", "troubleshooting", "comparison", "reference"
    ]
    followup_questions: list[str]


In [73]:
def rag_structured(query):
    search_results = index.search(query, num_results=5)
    prompt = build_prompt(query, search_results)

    response = openai_client.responses.parse(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": prompt}
        ],
        text_format=RAGResponse
    )
    return response


In [74]:
structured_response = rag_structured("python function definition")
structured_response.usage.input_tokens


6965

In [75]:
structured_response.usage.input_tokens - unstructured_response.usage.input_tokens


131